In [1]:
import pandas as pd
import os
import glob
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
# Define numbered_folders for use in subsequent cells
cases_dir = Path("31-08-2025/Cases")
numbered_folders = [f for f in cases_dir.iterdir() if f.is_dir() and f.name.isdigit()]
print(f"Found {len(numbered_folders)} numbered folders: {[f.name for f in numbered_folders]}")


Found 25 numbered folders: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '3', '4', '5', '6', '7', '8', '9']


In [3]:
# Read segmentation table files
def read_segmentation_tables(table_type):
    """
    Read all CSV files of a specific table type (arterial, late, venous)
    and combine them into a single dataframe
    """
    dataframes = []
    
    for folder in numbered_folders:
        segmenteringer_dir = folder / "Segmenteringer"
        if segmenteringer_dir.exists():
            table_file = segmenteringer_dir / f"table_{table_type}_{folder.name}.csv"
            if table_file.exists():
                try:
                    # Check if file is empty or has content
                    file_size = table_file.stat().st_size
                    if file_size == 0:
                        print(f"Warning: {table_file} is empty, skipping")
                        continue
                    
                    print(f"Reading {table_file} ({file_size} bytes)")
                    
                    # Try reading with different parameters to handle various CSV formats
                    try:
                        df = pd.read_csv(table_file, encoding='utf-8')
                    except UnicodeDecodeError:
                        try:
                            df = pd.read_csv(table_file, encoding='latin-1')
                        except:
                            df = pd.read_csv(table_file, encoding='cp1252')
                    
                    # Check if dataframe is empty
                    if df.empty:
                        print(f"Warning: {table_file} contains no data, skipping")
                        continue
                    
                    df['filename'] = table_file.name
                    df['case_number'] = folder.name
                    df['table_type'] = table_type
                    dataframes.append(df)
                    print(f"  Successfully loaded {len(df)} rows, {len(df.columns)} columns")
                    
                except pd.errors.EmptyDataError:
                    print(f"Warning: {table_file} is empty or has no columns, skipping")
                except pd.errors.ParserError as e:
                    print(f"Warning: {table_file} has parsing error: {e}, skipping")
                except Exception as e:
                    print(f"Error reading {table_file}: {e}, skipping")
            else:
                print(f"Warning: {table_file} not found")
        else:
            print(f"Warning: {segmenteringer_dir} not found for case {folder.name}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        print(f"\n{table_type.upper()} table combined dataframe shape: {combined_df.shape}")
        print(f"Columns: {list(combined_df.columns)}")
        print(f"Case numbers included: {sorted(combined_df['case_number'].unique())}")
        return combined_df
    else:
        print(f"No {table_type} table files found!")
        return None

# Read all three types of segmentation tables
print("=" * 50)
print("READING ARTERIAL TABLES")
print("=" * 50)
arterial_df = read_segmentation_tables("arterial")

print("\n" + "=" * 50)
print("READING LATE TABLES")
print("=" * 50)
late_df = read_segmentation_tables("late")

print("\n" + "=" * 50)
print("READING VENOUS TABLES")
print("=" * 50)
venous_df = read_segmentation_tables("venous")


READING ARTERIAL TABLES
Reading 31-08-2025\Cases\1\Segmenteringer\table_arterial_1.csv (751 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\10\Segmenteringer\table_arterial_10.csv (754 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\11\Segmenteringer\table_arterial_11.csv (763 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\12\Segmenteringer\table_arterial_12.csv (747 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\13\Segmenteringer\table_arterial_13.csv (769 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\14\Segmenteringer\table_arterial_14.csv (764 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\15\Segmenteringer\table_arterial_15.csv (758 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\16\Segmenteringer\table_arterial_16.csv (772 bytes)
  Successfully loaded 5 rows, 17 columns
Reading 31-08-2025\Cases\17\Segmen

In [4]:
# Display summary information about all dataframes
print("=" * 60)
print("SUMMARY OF ALL LOADED DATA")
print("=" * 60)

print(f"\n1. eGFR Data:")
if 'combined_df' in locals():
    print(f"   Shape: {combined_df.shape}")
    print(f"   Cases: {sorted(combined_df['case_number'].unique())}")

print(f"\n2. Arterial Tables:")
if arterial_df is not None:
    print(f"   Shape: {arterial_df.shape}")
    print(f"   Cases: {sorted(arterial_df['case_number'].unique())}")
    print(f"   Sample columns: {list(arterial_df.columns[:5])}")

print(f"\n3. Late Tables:")
if late_df is not None:
    print(f"   Shape: {late_df.shape}")
    print(f"   Cases: {sorted(late_df['case_number'].unique())}")
    print(f"   Sample columns: {list(late_df.columns[:5])}")

print(f"\n4. Venous Tables:")
if venous_df is not None:
    print(f"   Shape: {venous_df.shape}")
    print(f"   Cases: {sorted(venous_df['case_number'].unique())}")
    print(f"   Sample columns: {list(venous_df.columns[:5])}")

# Show first few rows of each dataframe for inspection
print("\n" + "=" * 60)
print("SAMPLE DATA PREVIEW")
print("=" * 60)

if 'combined_df' in locals():
    print("\n--- eGFR Data (first 3 rows) ---")
    print(combined_df.head(3))

if arterial_df is not None:
    print("\n--- Arterial Tables (first 3 rows) ---")
    print(arterial_df.head(3))

if late_df is not None:
    print("\n--- Late Tables (first 3 rows) ---")
    print(late_df.head(3))

if venous_df is not None:
    print("\n--- Venous Tables (first 3 rows) ---")
    print(venous_df.head(3))


SUMMARY OF ALL LOADED DATA

1. eGFR Data:

2. Arterial Tables:
   Shape: (105, 17)
   Cases: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '25', '3', '5', '7', '9']
   Sample columns: ['Segment', 'Voxel count (LM)', 'Volume mm3 (LM)', 'Volume cm3 (LM)', 'Voxel count (SV)']

3. Late Tables:
   Shape: (115, 17)
   Cases: ['10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '3', '4', '5', '7', '8', '9']
   Sample columns: ['Segment', 'Voxel count (LM)', 'Volume mm3 (LM)', 'Volume cm3 (LM)', 'Voxel count (SV)']

4. Venous Tables:
   Shape: (110, 17)
   Cases: ['10', '11', '12', '13', '14', '15', '16', '17', '2', '20', '21', '22', '23', '24', '25', '3', '4', '5', '6', '7', '8', '9']
   Sample columns: ['Segment', 'Voxel count (LM)', 'Volume mm3 (LM)', 'Volume cm3 (LM)', 'Voxel count (SV)']

SAMPLE DATA PREVIEW

--- Arterial Tables (first 3 rows) ---
                         Segment  Voxel count

In [5]:

# Set up the path to the Cases directory
cases_dir = Path("31-08-2025/Cases")

# Find all numbered subfolders
numbered_folders = [f for f in cases_dir.iterdir() if f.is_dir() and f.name.isdigit()]
print(f"Found {len(numbered_folders)} numbered folders: {[f.name for f in numbered_folders]}")

# Read all eGFR CSV files
dataframes = []
for folder in numbered_folders:
    eGFR_file = folder / f"eGFR_{folder.name}.csv"
    if eGFR_file.exists():
        print(f"Reading {eGFR_file}")
        df = pd.read_csv(eGFR_file, sep=';')
        df['filename'] = eGFR_file.name
        df['case_number'] = folder.name
        dataframes.append(df)
    else:
        print(f"Warning: {eGFR_file} not found")

# Combine all dataframes
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"\nCombined dataframe shape: {combined_df.shape}")
    print(f"Columns: {list(combined_df.columns)}")
    print(f"\nFirst few rows:")
    print(combined_df.head())
    
    print(f"\nCase numbers included: {sorted(combined_df['case_number'].unique())}")
else:
    print("No eGFR files found!")


Found 25 numbered folders: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '3', '4', '5', '6', '7', '8', '9']
Reading 31-08-2025\Cases\1\eGFR_1.csv
Reading 31-08-2025\Cases\10\eGFR_10.csv
Reading 31-08-2025\Cases\11\eGFR_11.csv
Reading 31-08-2025\Cases\12\eGFR_12.csv
Reading 31-08-2025\Cases\13\eGFR_13.csv
Reading 31-08-2025\Cases\14\eGFR_14.csv
Reading 31-08-2025\Cases\15\eGFR_15.csv
Reading 31-08-2025\Cases\16\eGFR_16.csv
Reading 31-08-2025\Cases\17\eGFR_17.csv
Reading 31-08-2025\Cases\18\eGFR_18.csv
Reading 31-08-2025\Cases\19\eGFR_19.csv
Reading 31-08-2025\Cases\2\eGFR_2.csv
Reading 31-08-2025\Cases\20\eGFR_20.csv
Reading 31-08-2025\Cases\21\eGFR_21.csv
Reading 31-08-2025\Cases\22\eGFR_22.csv
Reading 31-08-2025\Cases\23\eGFR_23.csv
Reading 31-08-2025\Cases\24\eGFR_24.csv
Reading 31-08-2025\Cases\25\eGFR_25.csv
Reading 31-08-2025\Cases\3\eGFR_3.csv
Reading 31-08-2025\Cases\4\eGFR_4.csv
Reading 31-08-2025\Cases\5\eGFR_5.csv
R

In [6]:
print(combined_df.shape)
print(combined_df.dtypes)
combined_df.head()

(1066, 109)
record_id                     int64
redcap_repeat_instrument     object
redcap_repeat_instance        int64
patient_id                  float64
patient_name                float64
                             ...   
gfr_value                    object
serum_creatinine            float64
gfr_complete                  int64
filename                     object
case_number                  object
Length: 109, dtype: object


,record_id,redcap_repeat_instrument,redcap_repeat_instance,patient_id,patient_name,cpr,date_of_birth,current_age,sex,height,weight,referral,scan_date,accession_number,excluded,exclusion_reason,demographics_complete,history_diabetes,dm_diagnosis_date,history_heart_disease,history_ischemic_heart_disease,history_heart_failure,history_hypertension,history_kidney_surgery,history_kidney_stones,history_ckd,history_hydronephrosis,history_malformation,history_kidney_injury,history_other,history_type_kidney_disease,medical_record_complete,contrast_agent,contrast_concentration,contrast_amount,contrast_rate,scanning_meta_data_complete,kidney_right_number,kidney_right_graft,kidney_right_artery_number,kidney_right_focal_lesion,kidney_right_cyst,kidney_right_calcification,kidney_right_cortex_defect,kidney_right_length,kidney_right_volume_parenchyma,kidney_right_sinus_fat,kidney_right_perirenal_fat,kidney_right_cortex_volume,kidney_left_number,kidney_left_graft,kidney_left_artery_number,kidney_left_focal_lesion,kidney_left_cyst,kidney_left_calcification,kidney_left_cortex_defect,kidney_left_length,kidney_left_volume_parenchyma_2,kidney_left_sinus_fat,kidney_left_cortex_volume,kidney_left_perirenal_fat,kidney_left_arterial_cortex,kidney_left_arterial_artery,kidney_left_arterial_vein,kidney_right_arterial_cortex,kidney_right_arterial_artery,kidney_right_arterial_vein,arterial_aorta,arterial_venacava_below_kidney,arterial_venecava_between_kidney_hepatic,arterial_venecava_above_hepatic,arterial_right_hepatic_vein,arterial_left_hepatic_vein,arterial_portal_vein,kidney_left_venous_cortex,kidney_right_venous_cortex,kidney_left_venous_artery,kidney_left_venous_vein,kidney_right_venous_artery,kidney_right_venous_vein,venous_aorta,venous_venacava_below_kidney,venous_venacava_between_kidney_hepatic,venous_venacava_above_hepatic,venous_right_hepatic_vein,venous_left_hepatic_vein,venous_portal_vein,kidney_left_late_cortex,kidney_right_late_cortex,kidney_left_late_artery,kidney_left_late_vein,kidney_right_late_vein,kidney_right_late_artery,late_aorta,late_venacava_below_kidney,late_venacava_between_kidney_hepatic,late_venacava_above_hepatic,late_right_hepatic_vein,late_rleft_hepatic_vein,late_portal_vein,pcct_measurements_complete,gfr_date,gfr_type,gfr_type_other,gfr_value,serum_creatinine,gfr_complete,filename,case_number
0,1,gfr,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29-04-2016,1,NaN,KOMM,NaN,2,eGFR_1.csv,1
1,1,gfr,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24-04-2017,1,NaN,83,86.0,2,eGFR_1.csv,1
2,1,gfr,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,09-04-2018,1,NaN,77,91.0,2,eGFR_1.csv,1
3,1,gfr,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [7]:
combined_df.describe()

,record_id,redcap_repeat_instance,patient_id,patient_name,cpr,date_of_birth,current_age,sex,height,weight,referral,scan_date,accession_number,excluded,exclusion_reason,demographics_complete,history_diabetes,dm_diagnosis_date,history_heart_disease,history_ischemic_heart_disease,history_heart_failure,history_hypertension,history_kidney_surgery,history_kidney_stones,history_ckd,history_hydronephrosis,history_malformation,history_kidney_injury,history_other,history_type_kidney_disease,medical_record_complete,contrast_agent,contrast_concentration,contrast_amount,contrast_rate,scanning_meta_data_complete,kidney_right_number,kidney_right_graft,kidney_right_artery_number,kidney_right_focal_lesion,kidney_right_cyst,kidney_right_calcification,kidney_right_cortex_defect,kidney_right_length,kidney_right_volume_parenchyma,kidney_right_sinus_fat,kidney_right_perirenal_fat,kidney_right_cortex_volume,kidney_left_number,kidney_left_graft,kidney_left_artery_number,kidney_left_focal_lesion,kidney_left_cyst,kidney_left_calcification,kidney_left_cortex_defect,kidney_left_length,kidney_left_volume_parenchyma_2,kidney_left_sinus_fat,kidney_left_cortex_volume,kidney_left_perirenal_fat,kidney_left_arterial_cortex,kidney_left_arterial_artery,kidney_left_arterial_vein,kidney_right_arterial_cortex,kidney_right_arterial_artery,kidney_right_arterial_vein,arterial_aorta,arterial_venacava_below_kidney,arterial_venecava_between_kidney_hepatic,arterial_venecava_above_hepatic,arterial_right_hepatic_vein,arterial_left_hepatic_vein,arterial_portal_vein,kidney_left_venous_cortex,kidney_right_venous_cortex,kidney_left_venous_artery,kidney_left_venous_vein,kidney_right_venous_artery,kidney_right_venous_vein,venous_aorta,venous_venacava_below_kidney,venous_venacava_between_kidney_hepatic,venous_venacava_above_hepatic,venous_right_hepatic_vein,venous_left_hepatic_vein,venous_portal_vein,kidney_left_late_cortex,kidney_right_late_cortex,kidney_left_late_artery,kidney_left_late_vein,kidney_right_late_vein,kidney_right_late_artery,late_aorta,late_venacava_below_kidney,late_venacava_between_kidney_hepatic,late_venacava_above_hepatic,late_right_hepatic_vein,late_rleft_hepatic_vein,late_portal_vein,pcct_measurements_complete,gfr_type,gfr_type_other,serum_creatinine,gfr_complete
count,1066.000000,1066.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1066.0,0.0,1059.000000,1066.0
mean,13.545966,36.329268,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,119.745042,2.0
std,6.773897,30.684544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,1367.942146,0.0
min,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [8]:
# Drop all columns that are fully empty (i.e., all values are NaN)
egfr_df = combined_df.dropna(axis=1, how='all').copy()

egfr_df.head()


,record_id,redcap_repeat_instrument,redcap_repeat_instance,gfr_date,gfr_type,gfr_value,serum_creatinine,gfr_complete,filename,case_number
0,1,gfr,1,29-04-2016,1,KOMM,NaN,2,eGFR_1.csv,1
1,1,gfr,2,24-04-2017,1,83,86.0,2,eGFR_1.csv,1
2,1,gfr,3,09-04-2018,1,77,91.0,2,eGFR_1.csv,1
3,1,gfr,4,07-05-2019,1,76,91.0,2,eGFR_1.csv,1
4,1,gfr,5,15-10-2020,1,69,99.0,2,eGFR_1.csv,1


In [9]:
egfr_df['date'] = pd.to_datetime(egfr_df['gfr_date'], format='%d-%m-%Y')
egfr_df.head()

,record_id,redcap_repeat_instrument,redcap_repeat_instance,gfr_date,gfr_type,gfr_value,serum_creatinine,gfr_complete,filename,case_number,date
0,1,gfr,1,29-04-2016,1,KOMM,NaN,2,eGFR_1.csv,1,2016-04-29
1,1,gfr,2,24-04-2017,1,83,86.0,2,eGFR_1.csv,1,2017-04-24
2,1,gfr,3,09-04-2018,1,77,91.0,2,eGFR_1.csv,1,2018-04-09
3,1,gfr,4,07-05-2019,1,76,91.0,2,eGFR_1.csv,1,2019-05-07
4,1,gfr,5,15-10-2020,1,69,99.0,2,eGFR_1.csv,1,2020-10-15


In [21]:
case_test = '3'

egfr_df[egfr_df['case_number'] == case_test].sort_values(by='date', ascending=False)

,record_id,redcap_repeat_instrument,redcap_repeat_instance,gfr_date,gfr_type,gfr_value,serum_creatinine,gfr_complete,filename,case_number,date
902,3,gfr,12,19-02-2025,1,>90,67.0,2,eGFR_3.csv,3,2025-02-19
901,3,gfr,11,01-11-2024,1,>90,73.0,2,eGFR_3.csv,3,2024-11-01
900,3,gfr,10,31-10-2024,1,>90,65.0,2,eGFR_3.csv,3,2024-10-31
899,3,gfr,9,25-09-2024,1,>90,67.0,2,eGFR_3.csv,3,2024-09-25
898,3,gfr,8,28-06-2022,1,84,88.0,2,eGFR_3.csv,3,2022-06-28
897,3,gfr,7,19-11-2020,1,>90,66.0,2,eGFR_3.csv,3,2020-11-19
896,3,gfr,6,08-09-2020,1,>90,77.0,2,eGFR_3.csv,3,2020-09-08
895,3,gfr,5,06-01-2019,1,>90,83.0,2,eGFR_3.csv,3,2019-01-06
894,3,gfr,4,08-03-2018,1,>90,83.0,2,eGFR_3.csv,3,2018-03-08
893,3,gfr,3,27-06-2017,1,>90,74.0,2,eGFR_3.csv,3,2017-06-27


In [22]:
arterial_df[arterial_df['case_number'] == case_test]

,Segment,Voxel count (LM),Volume mm3 (LM),Volume cm3 (LM),Voxel count (SV),Volume mm3 (SV),Volume cm3 (SV),Minimum,Maximum,Mean,Standard deviation,Percentile 5,Percentile 95,Median,filename,case_number,table_type
85,arterial_threshold_3,1544117,1048210.00,1048.21000,1544117,1048210.00,1048.21000,173,981,283.903,101.1590,180,508,256,table_arterial_3.csv,3,arterial
86,arterial_renal_artery_right_3,1715,1164.21,1.16421,1715,1164.21,1.16421,173,335,244.731,38.7353,180,300,247,table_arterial_3.csv,3,arterial
87,arterial_renal_artery_left_3,1597,1084.11,1.08411,1597,1084.11,1.08411,173,355,250.023,39.4441,181,310,255,table_arterial_3.csv,3,arterial
88,arterial_renal_vein_right_3,1640,1113.30,1.11330,1640,1113.30,1.11330,173,295,243.729,23.1916,191,271,248,table_arterial_3.csv,3,arterial
89,arterial_renal_vein_left_3,4770,3238.07,3.23807,4770,3238.07,3.23807,173,308,243.719,23.8056,191,274,248,table_arterial_3.csv,3,arterial


In [23]:
venous_df[venous_df['case_number'] == case_test]

,Segment,Voxel count (LM),Volume mm3 (LM),Volume cm3 (LM),Voxel count (SV),Volume mm3 (SV),Volume cm3 (SV),Minimum,Maximum,Mean,Standard deviation,Percentile 5,Percentile 95,Median,filename,case_number,table_type
75,venous_threshold_3,3282836,2795200.000,2795.200000,3282836,2795200.000,2795.200000,108,981,240.381,136.9520,115,540,187,table_venous_3.csv,3,venous
76,venous_renal_artery_right_3,915,779.086,0.779086,915,779.086,0.779086,108,183,133.383,16.7119,109,161,131,table_venous_3.csv,3,venous
77,venous_renal_artery_left_3,913,777.383,0.777383,913,777.383,0.777383,108,206,136.653,21.0364,109,176,132,table_venous_3.csv,3,venous
78,venous_renal_vein_right_3,1054,897.439,0.897439,1054,897.439,0.897439,108,189,138.340,13.8400,115,160,137,table_venous_3.csv,3,venous
79,venous_renal_vein_left_3,3193,2718.710,2.718710,3193,2718.710,2.718710,108,193,136.091,14.9250,111,161,135,table_venous_3.csv,3,venous


In [24]:
late_df[late_df['case_number'] == case_test]

,Segment,Voxel count (LM),Volume mm3 (LM),Volume cm3 (LM),Voxel count (SV),Volume mm3 (SV),Volume cm3 (SV),Minimum,Maximum,Mean,Standard deviation,Percentile 5,Percentile 95,Median,filename,case_number,table_type
85,late_threshold_3,4818231,3270810.00,3270.81000,4818231,3270810.00,3270.81000,28,981,97.5205,112.0250,28,346,50,table_late_3.csv,3,late
86,late_renal_artery_right_3,1793,1217.16,1.21716,1793,1217.16,1.21716,28,122,52.9955,17.4200,29,85,49,table_late_3.csv,3,late
87,late_renal_artery_left_3,1984,1346.82,1.34682,1984,1346.82,1.34682,28,128,57.4259,17.4288,30,87,55,table_late_3.csv,3,late
88,late_renal_vein_right_3,1665,1130.27,1.13027,1665,1130.27,1.13027,28,97,53.5315,13.5851,31,75,52,table_late_3.csv,3,late
89,late_renal_vein_left_3,5382,3653.52,3.65352,5382,3653.52,3.65352,28,120,62.4026,17.5732,32,90,62,table_late_3.csv,3,late


In [25]:
"""
Create iodine concentration table from segmentation data.

This script reads the arterial, venous, and late segmentation tables
and creates a combined table with:
- case_number
- kidney_side (left/right)
- phase (arterial/venous/late)
- iodine_concentration (Mean value)
"""

import pandas as pd
from pathlib import Path

# Set up the path to the Cases directory
cases_dir = Path("31-08-2025/Cases")

# Find all numbered subfolders
numbered_folders = [f for f in cases_dir.iterdir() if f.is_dir() and f.name.isdigit()]
print(f"Found {len(numbered_folders)} numbered folders")

# Function to read segmentation tables
def read_segmentation_tables(table_type):
    """Read all CSV files of a specific table type and combine them."""
    dataframes = []
    
    for folder in numbered_folders:
        segmenteringer_dir = folder / "Segmenteringer"
        if segmenteringer_dir.exists():
            table_file = segmenteringer_dir / f"table_{table_type}_{folder.name}.csv"
            if table_file.exists():
                try:
                    file_size = table_file.stat().st_size
                    if file_size <= 2:  # Skip empty files
                        continue
                    
                    df = pd.read_csv(table_file, encoding='utf-8')
                    
                    if df.empty:
                        continue
                    
                    df['filename'] = table_file.name
                    df['case_number'] = folder.name
                    df['table_type'] = table_type
                    dataframes.append(df)
                    
                except (pd.errors.EmptyDataError, pd.errors.ParserError):
                    continue
                except Exception as e:
                    print(f"Error reading {table_file}: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        return combined_df
    else:
        return None

# Read all three types of segmentation tables
print("\nReading segmentation tables...")
arterial_df = read_segmentation_tables("arterial")
venous_df = read_segmentation_tables("venous")
late_df = read_segmentation_tables("late")

print(f"Arterial: {len(arterial_df) if arterial_df is not None else 0} rows")
print(f"Venous: {len(venous_df) if venous_df is not None else 0} rows")
print(f"Late: {len(late_df) if late_df is not None else 0} rows")

# Function to extract kidney data from segmentation tables
def extract_kidney_data(df, phase_name):
    """
    Extract kidney iodine concentration data from segmentation tables.
    
    Parameters:
    - df: DataFrame with segmentation data
    - phase_name: Name of the phase ('arterial', 'venous', or 'late')
    
    Returns:
    - DataFrame with columns: case_number, kidney_side, phase, iodine_concentration
    """
    if df is None:
        return pd.DataFrame(columns=['case_number', 'kidney_side', 'phase', 'iodine_concentration'])
    
    records = []
    
    for _, row in df.iterrows():
        segment = str(row['Segment']).lower()
        
        # Skip threshold segments
        if 'threshold' in segment:
            continue
        
        # Check if this is a kidney/renal segment (artery or vein)
        if 'renal' in segment or 'kidney' in segment:
            # Determine kidney side (left or right)
            if 'left' in segment:
                kidney_side = 'left'
            elif 'right' in segment:
                kidney_side = 'right'
            else:
                continue  # Skip if side cannot be determined
            
            records.append({
                'case_number': row['case_number'],
                'kidney_side': kidney_side,
                'phase': phase_name,
                'iodine_concentration': row['Mean']
            })
    
    return pd.DataFrame(records)

# Extract data from each phase
print("\nExtracting kidney data from each phase...")
arterial_kidney = extract_kidney_data(arterial_df, 'arterial')
venous_kidney = extract_kidney_data(venous_df, 'venous')
late_kidney = extract_kidney_data(late_df, 'late')

print(f"Extracted {len(arterial_kidney)} arterial kidney measurements")
print(f"Extracted {len(venous_kidney)} venous kidney measurements")
print(f"Extracted {len(late_kidney)} late kidney measurements")

# Combine all phases
iodine_concentration_table = pd.concat([arterial_kidney, venous_kidney, late_kidney], ignore_index=True)

# Sort by case_number, kidney_side, and phase for better readability
iodine_concentration_table = iodine_concentration_table.sort_values(
    by=['case_number', 'kidney_side', 'phase']
).reset_index(drop=True)

# Display summary
print("\n" + "="*60)
print("IODINE CONCENTRATION TABLE CREATED")
print("="*60)
print(f"Total rows: {len(iodine_concentration_table)}")
print(f"Columns: {list(iodine_concentration_table.columns)}")
print(f"\nUnique cases: {sorted(iodine_concentration_table['case_number'].unique())}")
print(f"Phases: {sorted(iodine_concentration_table['phase'].unique())}")
print(f"Kidney sides: {sorted(iodine_concentration_table['kidney_side'].unique())}")

# Save to CSV
output_file = "iodine_concentration_table.csv"
iodine_concentration_table.to_csv(output_file, index=False)
print(f"\n✓ Table saved to: {output_file}")

# Display first 20 rows
iodine_concentration_table.head()


Found 25 numbered folders

Reading segmentation tables...
Arterial: 105 rows
Venous: 110 rows
Late: 115 rows

Extracting kidney data from each phase...
Extracted 84 arterial kidney measurements
Extracted 88 venous kidney measurements
Extracted 92 late kidney measurements

IODINE CONCENTRATION TABLE CREATED
Total rows: 264
Columns: ['case_number', 'kidney_side', 'phase', 'iodine_concentration']

Unique cases: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '3', '4', '5', '6', '7', '8', '9']
Phases: ['arterial', 'late', 'venous']
Kidney sides: ['left', 'right']

✓ Table saved to: iodine_concentration_table.csv


,case_number,kidney_side,phase,iodine_concentration
0,1,left,arterial,174.652
1,1,left,arterial,121.047
2,1,right,arterial,183.555
3,1,right,arterial,135.824
4,10,left,arterial,214.515


In [26]:
iodine_concentration_table

,case_number,kidney_side,phase,iodine_concentration
0,1,left,arterial,174.6520
1,1,left,arterial,121.0470
2,1,right,arterial,183.5550
3,1,right,arterial,135.8240
4,10,left,arterial,214.5150
...,...,...,...,...
259,9,right,arterial,194.4050
260,9,right,late,66.1439
261,9,right,late,72.9041
262,9,right,venous,133.3460
